# Experiment: Case-001 Doctor Response Triage Gate

Objective: define a public-safe triage map for clinician replies without storing raw doctor messages or patient-specific treatment instructions in the public repo.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class ResponseDomain:
    """Public-safe routing rule for one clinician-response domain."""

    name: str
    current_label: str
    priority: int
    allowed_outcomes: tuple[str, ...]
    public_update_allowed_without_release_check: bool

In [ ]:
OUTCOMES = (
    "confirmed",
    "corrected",
    "not_confirmable_missing_records",
    "not_applicable",
    "under_specialist_review",
)

domains = [
    ResponseDomain("diagnosis", "phenotype_only", 5, OUTCOMES, False),
    ResponseDomain(
        "clinical_dependence",
        "transfusion_dependent_reported",
        5,
        OUTCOMES,
        False,
    ),
    ResponseDomain(
        "transfusion_burden",
        "transfusion_dependent_burden_unquantified",
        5,
        OUTCOMES,
        False,
    ),
    ResponseDomain(
        "immune_blood_bank",
        "immune_transfusion_packet_missing",
        4,
        OUTCOMES,
        False,
    ),
    ResponseDomain("iron_organ_chelation", "iron_packet_missing", 4, OUTCOMES, False),
    ResponseDomain(
        "referral_readiness",
        "advanced_therapy_referral_packet_missing",
        4,
        OUTCOMES,
        False,
    ),
]

triage_order = [
    domain.name for domain in sorted(domains, key=lambda item: -item.priority)
]
blocked_public_updates = [
    domain.name
    for domain in domains
    if not domain.public_update_allowed_without_release_check
]

decision = {
    "gate_label": "doctor_response_triage_ready",
    "current_state": "no_doctor_response_recorded",
    "triage_order": triage_order,
    "allowed_outcomes": list(OUTCOMES),
    "public_updates_blocked_until_release_check": blocked_public_updates,
    "patient_specific_claims": "blocked_until_clinician_response_is_deidentified",
}

decision

## Result

A doctor reply should be routed domain by domain. No public case update is allowed until the response is de-identified and passed through the public release checklist.
